In [1]:
# HELPER FUNCTIONS

# coord_name() allows the script to recognise common latitude/longitude/time
# coordinate naming conventions instead of assuming a single NetCDF convention.
def coord_name(ds, names):
    for name in names:
        if name in ds.coords or name in ds.dims:
            return name
    raise ValueError(f"Cannot identify coordinate from {names}")


def detrend_matrix(x):
    nt = x.shape[0]
    t = np.arange(nt, dtype=np.float64)
    tc = t - t.mean()
    xm = x.mean(axis=0)
    slope = np.sum(tc[:, None] * (x - xm[None, :]), axis=0) / np.sum(tc**2)
    return x - tc[:, None] * slope[None, :], slope


def scree_elbow(evals):
    """Maximum-distance elbow in log-eigenvalue space."""
    if len(evals) <= 2:
        return 1
    x = np.arange(1, len(evals)+1, dtype=float)
    y = np.log(np.maximum(evals, np.finfo(float).tiny))
    x1, y1 = x[0], y[0]
    x2, y2 = x[-1], y[-1]
    d = np.abs((y2-y1)*x - (x2-x1)*y + x2*y1 - y2*x1)
    d /= np.sqrt((y2-y1)**2 + (x2-x1)**2)
    return int(np.argmax(d) + 1)


def first_threshold(cum, threshold):
    hit = np.where(cum >= threshold)[0]
    return None if len(hit) == 0 else int(hit[0] + 1)

In [2]:
import os
import warnings
import numpy as np
import pandas as pd
import xarray as xr
from scipy.sparse.linalg import svds

# 1A. OPEN DATASET AND IDENTIFY COORDINATES

sst = xr.open_dataset("/home/26n5005/climate-data/sst.mon.mean_OISSTv2p1_IndoPacific_lon20E-290E_lat30S-30N_198109-202607.nc")
mswep = xr.open_dataset("/home/26n5005/climate-data/mswep_pr_monthly_brunei_1979_2026.nc")

time = coord_name(sst, ["time"])
latn = coord_name(sst, ["lat", "latitude", "y"])
lonn = coord_name(sst, ["lon", "longitude", "x"])

print(sst.dims)
print(sst.coords)
print(sst.data_vars)

print(mswep.dims)
print(mswep.coords)
print(mswep.data_vars)

FrozenMappingWarningOnValuesAccess({'time': 539, 'lat': 240, 'lon': 1080})
Coordinates:
  * time     (time) datetime64[ns] 4kB 1981-09-01 1981-10-01 ... 2026-07-01
  * lat      (lat) float32 960B -29.88 -29.62 -29.38 ... 29.38 29.62 29.88
  * lon      (lon) float32 4kB 20.12 20.38 20.62 20.88 ... 289.4 289.6 289.9
Data variables:
    sst      (time, lat, lon) float32 559MB ...
FrozenMappingWarningOnValuesAccess({'time': 571, 'bnds': 2, 'lat': 13, 'lon': 15})
Coordinates:
  * time     (time) datetime64[ns] 5kB 1979-02-14 1979-03-16 ... 2026-08-16
  * lat      (lat) float32 52B 5.25 5.15 5.05 4.95 4.85 ... 4.35 4.25 4.15 4.05
  * lon      (lon) float32 60B 114.1 114.2 114.3 114.4 ... 115.3 115.4 115.4
Data variables:
    time_bnds      (time, bnds) datetime64[ns] 9kB ...
    precipitation  (time, lat, lon) float32 445kB ...


In [3]:
# changing the date-time to yyyy-mm format
sst["time"] = sst.time.values.astype("datetime64[M]")
mswep["time"] = mswep.time.values.astype("datetime64[M]")

# common months only
common_months = sorted(set(sst.time.values) & set(mswep.time.values))

sst = sst.sel(time=common_months)
mswep = mswep.sel(time=common_months)

In [4]:
# Restrict both datasets to the common study period
START_DATE = np.datetime64("1981-09")
END_DATE = np.datetime64("2025-12")
sst = sst.sel(time=slice(START_DATE, END_DATE))

sst = sst["sst"].squeeze(drop=True).transpose("time", "lat", "lon")
sst = sst.where(np.isfinite(sst))

nt, nlat, nlon = sst.sizes["time"], sst.sizes["lat"], sst.sizes["lon"]

print(f"Period    : {str(sst.time.values[0])[:10]} to {str(sst.time.values[-1])[:10]}")
print(f"Time steps: {nt}")
print(f"Grid      : {nlat} x {nlon}")

Period    : 1981-09-01 to 2025-12-01
Time steps: 532
Grid      : 240 x 1080


In [5]:
# 1B. FORM THE RAW TIME x SPACE MATRIX AND DEFINE THE OCEAN MASK

raw_full = sst.values.reshape(nt, nlat*nlon).astype(np.float64)
valid = np.all(np.isfinite(raw_full), axis=0)
raw = raw_full[:, valid]
if raw.shape[1] == 0:
    raise RuntimeError("No grid cells have complete temporal coverage.")
    
print(f"Valid cells: {raw.shape[1]:,}")

Valid cells: 204,985


In [ ]:
# 1C. REMOVE GRID-POINT LINEAR TRENDS

detrended, slope = detrend_matrix(raw)

detrended_full = np.full((nt, nlat*nlon), np.nan, dtype=np.float64)
detrended_full[:, valid] = detrended
detrended_da = xr.DataArray(
    detrended_full.reshape(nt, nlat, nlon),
    dims=(time, latn, lonn),
    coords={time: sst[time], latn: sst[latn], lonn: sst[lonn]},
    name="sst_detrended",
    attrs={"long_name": "linearly detrended SST", "units": sst.attrs.get("units", "degC")},
)

In [7]:
# 1D. COMPUTE THE 12-MONTH CLIMATOLOGY FROM DETRENDED SST

clim = detrended_da.groupby(f"{time}.month").mean(time, skipna=True)
clim.attrs["processing_order"] = "linear detrending before monthly climatology"
clim.to_dataset(name="sst_climatology_detrended").to_netcdf(
    "sst_climatology_detrended.nc"
)

In [8]:
# 1E. REMOVE THE MONTHLY CLIMATOLOGY TO FORM SST ANOMALIES

anom = detrended_da.groupby(f"{time}.month") - clim
X = anom.values.reshape(nt, nlat*nlon)[:, valid].astype(np.float64)
X -= X.mean(axis=0, keepdims=True)

In [9]:
# 1F. APPLY LATITUDE/AREA WEIGHTING

lat = sst[latn].values.astype(float)
wlat = np.sqrt(np.clip(np.cos(np.deg2rad(lat)), 0.0, None))
w2d = np.repeat(wlat[:, None], nlon, axis=1).reshape(-1)
w = w2d[valid]
Xw = X * w[None, :]

total_var = np.sum(Xw**2) / (nt - 1)
print(f"Weighted matrix shape : {Xw.shape}")
print(f"Total weighted variance: {total_var:.6g}")

# Free large arrays that are no longer needed before SVD. This reduces memory
# pressure, which is important for a 0.25-degree Indo-Pacific SST domain.
# release large temporaries before SVD
for name in ["raw_full", "detrended_full"]:
    if name in globals():
        del globals()[name]

Weighted matrix shape : (539, 204985)
Total weighted variance: 69155.6


In [10]:
# 2nd stage: TRAIN-TEST SPLIT

TEST_MONTHS = 60
nt = Xw.shape[0]
split = nt - TEST_MONTHS

Xw_train = Xw[:split]
Xw_test = Xw[split:]

nt_train = Xw_train.shape[0]
nt_test = Xw_test.shape[0]

print(f"Train shape: {Xw_train.shape}")
print(f"Test shape : {Xw_test.shape}")
print(f"Train months: {nt_train}")
print(f"Test months : {nt_test}")

Train shape: (479, 204985)
Test shape : (60, 204985)
Train months: 479
Test months : 60


In [11]:
# 3rd stage: computing SVD (on train only)

NMODES=10 
K_TRUNC=30

k = min(K_TRUNC, min(Xw_train.shape)-1)
if k < NMODES:
    raise RuntimeError(f"Only {k} SVD modes available but NMODES={NMODES}")

U_train, s, VT = svds(Xw_train, k=k, which="LM", return_singular_vectors=True)
idx = np.argsort(s)[::-1]
s, U_train, VT = s[idx], U_train[:, idx], VT[idx, :]
print(f"Computed leading {k} singular triplets.")

Computed leading 30 singular triplets.


In [12]:
# 4th stage: CONSTRUCT PCs AND EOF LOADINGS

PC_train = U_train * s[None, :]
PC_test = Xw_test @ VT.T

pc_mean = PC_train.mean(axis=0)
pc_sd = PC_train.std(axis=0, ddof=1)
pc_sd[pc_sd == 0] = 1.0

# note that for test PC is estimated on train data. That is to avoid data leackage.
PC_train_z = (PC_train - pc_mean[None, :]) / pc_sd[None, :]
PC_test_z = (PC_test - pc_mean[None, :]) / pc_sd[None, :]

EOFw = VT * pc_sd[:, None]
EOF_valid = EOFw / w[None, :]

# Reinsert the valid-ocean EOF values into the full latitude-longitude grid,
# leaving land/missing cells as NaN. Only the requested NMODES are retained.
EOF_full = np.full((NMODES, nlat*nlon), np.nan, dtype=np.float32)
EOF_full[:, valid] = EOF_valid[:NMODES].astype(np.float32)
EOF = EOF_full.reshape(NMODES, nlat, nlon)

PC_train_out = PC_train_z[:, :NMODES].astype(np.float32)
PC_test_out = PC_test_z[:, :NMODES].astype(np.float32)

print(f"Saved EOF modes: {NMODES}")
print(f"EOF shape      : {EOF.shape}")
print(f"Train PC shape  : {PC_train_out.shape}")
print(f"Test PC shape   : {PC_test_out.shape}")

Saved EOF modes: 10
EOF shape      : (10, 240, 1080)
Train PC shape  : (479, 10)
Test PC shape   : (60, 10)


In [13]:
# 5th stage: EIGENVALUES AND EXPLAINED VARIANCE

total_var_train = np.sum(Xw_train**2) / (nt_train - 1)

eigenvalue = s**2 / (nt_train - 1)
evf = eigenvalue / total_var_train
cum = np.cumsum(evf)

north = eigenvalue * np.sqrt(2.0/nt_train)
sep = np.ones(k, dtype=bool)
sep[:-1] = (eigenvalue[:-1] - eigenvalue[1:]) > north[:-1]

In [14]:
# TRUNCATION DIAGNOSTICS (train only)

# Use several complementary diagnostics rather than a single arbitrary cutoff:
# automatic scree elbow, 80% and 90% cumulative variance, and North separation.
k_elbow = scree_elbow(eigenvalue)
k80 = first_threshold(cum, 0.80)
k90 = first_threshold(cum, 0.90)
k80txt = str(k80) if k80 is not None else f">{k}"
k90txt = str(k90) if k90 is not None else f">{k}"
elbow_sep = bool(sep[k_elbow-1]) if k_elbow < k else True

for i in range(min(k, 10)):
    print(f"EOF{i+1:02d}: EV={100*evf[i]:7.3f}%  cumulative={100*cum[i]:7.3f}%")

print("\nTruncation diagnostics")
print(f"Scree elbow             : EOF {k_elbow}")
print(f">=80% cumulative variance: {k80txt}")
print(f">=90% cumulative variance: {k90txt}")
print(f"North separation at elbow: {'YES' if elbow_sep else 'NO'}")
print(f"\nPROPOSED TRUNCATION: approximately EOF1-EOF{k_elbow} based on the scree elbow.")
if not elbow_sep:
    print("CAUTION: the elbow boundary is not clearly separated by North's rule; inspect the adjacent EOF pair jointly.")

modes = np.arange(1, k+1)
# Save a machine-readable table for every SVD mode inspected so the truncation
# decision can be revisited without rerunning the expensive decomposition.
pd.DataFrame({
    "mode": modes,
    "singular_value": s,
    "eigenvalue": eigenvalue,
    "explained_variance_fraction": evf,
    "explained_variance_percent": 100*evf,
    "cumulative_variance_fraction": cum,
    "cumulative_variance_percent": 100*cum,
    "north_sampling_error": north,
    "north_separated_from_next": sep,
}).to_csv("sst_eof_variance_truncation.csv", index=False)

# Also save a concise human-readable truncation recommendation.
with open("sst_eof_truncation_recommendation.txt", "w", encoding="utf-8") as f:
    f.write("SST EOF TRUNCATION DIAGNOSTICS\n")
    f.write("="*72 + "\n")
    f.write(f"Monthly samples: {nt}\n")
    f.write(f"SVD modes inspected: {k}\n\n")
    f.write(f"Scree elbow: EOF {k_elbow}\n")
    f.write(f"Modes for >=80% cumulative variance: {k80txt}\n")
    f.write(f"Modes for >=90% cumulative variance: {k90txt}\n")
    f.write(f"North separation at elbow: {'YES' if elbow_sep else 'NO'}\n\n")
    f.write(f"PROPOSED TRUNCATION: EOF1-EOF{k_elbow}, subject to physical interpretation and North-rule degeneracy.\n")

EOF01: EV= 27.685%  cumulative= 27.685%
EOF02: EV=  7.517%  cumulative= 35.202%
EOF03: EV=  4.444%  cumulative= 39.645%
EOF04: EV=  3.849%  cumulative= 43.494%
EOF05: EV=  2.886%  cumulative= 46.381%
EOF06: EV=  2.279%  cumulative= 48.659%
EOF07: EV=  2.008%  cumulative= 50.668%
EOF08: EV=  1.745%  cumulative= 52.413%
EOF09: EV=  1.649%  cumulative= 54.062%
EOF10: EV=  1.547%  cumulative= 55.609%

Truncation diagnostics
Scree elbow             : EOF 6
>=80% cumulative variance: >30
>=90% cumulative variance: >30
North separation at elbow: YES

PROPOSED TRUNCATION: approximately EOF1-EOF6 based on the scree elbow.


In [16]:
# writing EOF/PC results to NETCDF

mode_out = np.arange(1, NMODES+1, dtype=np.int32)

time_train = sst[time].isel({time: slice(0, split)})
time_test = sst[time].isel({time: slice(split, None)})

dsout = xr.Dataset(
    data_vars={
        "EOF": (("mode", latn, lonn), EOF, {"long_name": "SST EOF spatial loading"}),
        
        "PC_train": (("time_train", "mode"), PC_train_out, {"long_name": "standardised training principal components"}),
        "PC_test": (("time_test", "mode"), PC_test_out, {"long_name": "standardised test principal components"}),

        "eigenvalue": (("mode",), eigenvalue[:NMODES]),
        "explained_variance_fraction": (("mode",), evf[:NMODES]),
        "cumulative_explained_variance_fraction": (("mode",), cum[:NMODES]),
        "north_sampling_error": (("mode",), north[:NMODES]),
        "valid_ocean_mask": ((latn, lonn), valid.reshape(nlat,nlon).astype(np.int8)),
        "sqrt_coslat_weight": ((latn,), wlat.astype(np.float32)),
    },
    coords={"mode": mode_out, "time_train": time_train.values, "time_test": time_test.values, latn: sst[latn], lonn: sst[lonn]},
    attrs={
        "title": "EOF analysis of detrended monthly SST",
        "method": "Singular Value Decomposition (SVD)",
        "processing_order": "detrend -> monthly climatology -> anomalies -> sqrt(cos(latitude)) weighting -> SVD",
        "scree_elbow_mode": int(k_elbow),
        "north_separation_at_elbow": str(elbow_sep),
        "note": "EOF/PC signs are mathematically arbitrary.",
    },
)

dsout.to_netcdf(
    "sst_eof_svd_results.nc",
    encoding={
        "EOF": {"zlib": True, "complevel": 4, "dtype": "float32"},
        "PC_train": {"zlib": True, "complevel": 4, "dtype": "float32"},
        "PC_test": {"zlib": True, "complevel": 4, "dtype": "float32"},
    },
)

In [17]:
# PLOT THE FIRST THREE EOF/PC PAIRS

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAVE_CARTOPY = True
except ImportError:
    HAVE_CARTOPY = False
    print("WARNING: cartopy unavailable; plotting plain lon-lat EOF panels.")

nplot = min(3, NMODES)

# Separate train and test dates
all_dates = sst[time].values
train_dates = all_dates[:split]
test_dates = all_dates[split:]

fig = plt.figure(figsize=(16, 12))

for i in range(nplot):
    data = EOF[i]
    maxabs = np.nanmax(np.abs(data))

    if not np.isfinite(maxabs) or maxabs == 0:
        maxabs = 1.0

    levels = np.linspace(-maxabs, maxabs, 21)

    # Longitude in 0-360 degrees east
    lon_plot = np.mod(sst[lonn].values.astype(float), 360.0)
    lat_plot = sst[latn].values.astype(float)

    lon_order = np.argsort(lon_plot)
    lon_plot = lon_plot[lon_order]
    data_plot = data[:, lon_order]

    lon_min, lon_max = float(lon_plot.min()), float(lon_plot.max())
    lat_min, lat_max = float(lat_plot.min()), float(lat_plot.max())

    candidate_xticks = np.arange(0.0, 361.0, 30.0)
    xticks = candidate_xticks[
        (candidate_xticks >= lon_min - 1e-6) &
        (candidate_xticks <= lon_max + 1e-6)
    ]

    # ------------------------------------------------------------------
    # Left panel: EOF spatial pattern
    # ------------------------------------------------------------------
    if HAVE_CARTOPY:
        map_crs = ccrs.PlateCarree(central_longitude=180.0)
        data_crs = ccrs.PlateCarree()

        axm = fig.add_subplot(3, 2, 2*i+1, projection=map_crs)
        cf = axm.contourf(
            lon_plot, lat_plot, data_plot,
            levels=levels, cmap="RdBu_r", extend="both",
            transform=data_crs
        )

        axm.set_extent(
            [lon_min, lon_max, lat_min, lat_max],
            crs=data_crs
        )
        axm.coastlines(linewidth=0.8)
        axm.add_feature(
            cfeature.LAND,
            facecolor="lightgray",
            zorder=3
        )

        axm.gridlines(
            crs=data_crs,
            draw_labels=False,
            xlocs=xticks,
            linewidth=0.4,
            alpha=0.5,
            linestyle="--"
        )

        axm.set_xticks(xticks, crs=data_crs)
        axm.set_xticklabels([f"{int(round(x))}°E" for x in xticks])

        yticks = np.arange(
            np.ceil(lat_min / 10.0) * 10.0,
            np.floor(lat_max / 10.0) * 10.0 + 0.1,
            10.0
        )
        axm.set_yticks(yticks, crs=data_crs)
        axm.set_yticklabels([
            "0°" if abs(y) < 1e-9
            else f"{abs(int(round(y)))}°{'N' if y > 0 else 'S'}"
            for y in yticks
        ])

    else:
        axm = fig.add_subplot(3, 2, 2*i+1)
        cf = axm.contourf(
            lon_plot, lat_plot, data_plot,
            levels=levels, cmap="RdBu_r", extend="both"
        )
        axm.set_xlim(lon_min, lon_max)
        axm.set_xticks(xticks)
        axm.set_xticklabels([f"{int(round(x))}°E" for x in xticks])
        axm.set_xlabel("Longitude")
        axm.set_ylabel("Latitude")

    axm.set_title(
        f"EOF{i+1} — {100*evf[i]:.2f}% variance",
        fontweight="bold"
    )
    cb = fig.colorbar(
        cf, ax=axm, orientation="horizontal",
        pad=0.08, fraction=0.05
    )
    cb.set_label("EOF loading (°C per 1 SD of PC)")

    # ------------------------------------------------------------------
    # Right panel: standardised PC, with train/test split
    # ------------------------------------------------------------------
    axp = fig.add_subplot(3, 2, 2*i+2)

    axp.plot(
        train_dates,
        PC_train_out[:, i],
        linewidth=0.8,
        label="Train"
    )
    axp.plot(
        test_dates,
        PC_test_out[:, i],
        linewidth=0.8,
        label="Test"
    )

    axp.axvline(
        test_dates[0],
        color="black",
        linewidth=1.0,
        linestyle="--",
        label="Train/test split"
    )
    axp.axhline(0.0, color="gray", linewidth=0.7, linestyle="--")

    axp.set_title(f"PC{i+1}", fontweight="bold")
    axp.set_ylabel("Standardised PC")
    axp.grid(True, linewidth=0.4, alpha=0.5)
    axp.legend(fontsize=8)

    if i == nplot - 1:
        axp.set_xlabel("Time")

fig.suptitle(
    "Leading EOFs and PCs of Indo-Pacific OISSTv2.1\n"
    "EOFs fitted on training data; test PCs projected onto the same EOF basis",
    fontsize=14,
    fontweight="bold",
    y=0.995
)

fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig("EOF1-3_PC1-3.png", dpi=200, bbox_inches="tight")
plt.close(fig)

print(f'Figure saved to: "EOF1-3_PC1-3.png"')

/home/gridtf/cartopy/io/__init__.py:263: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/110m_physical/ne_110m_land.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)


Figure saved to: "EOF1-3_PC1-3.png"


In [19]:
# SCREE PLOT

fig, ax = plt.subplots(figsize=(9,5.5))
ax.plot(modes, 100*evf, marker="o", markersize=4, linewidth=1.0)
ax.axvline(k_elbow, linestyle="--", linewidth=1.0, label=f"Scree elbow = EOF {k_elbow}")
ax.set_xlabel("EOF mode")
ax.set_ylabel("Explained variance (%)")
ax.set_title("EOF Scree Plot and Proposed Truncation")
ax.grid(True, linewidth=0.4, alpha=0.5)
ax.legend()
fig.tight_layout()
fig.savefig("sst_eof_scree_truncation.png", dpi=200, bbox_inches="tight")
plt.close(fig)
